In [1]:
import os
import sys
sys.path.append("/users/ziyzhang/topology-research/")
from topologies.DDF import DDFtopo
import globals as gl
import numpy as np
import csv
from nexullance.ultility import nexullance_exp_container
# import pickle

Cap_core = 10 #GBps
Cap_access = 10 #GBps

In [2]:
config = gl.ddf_configs[0]
V = config[0]
D = config[1]
EPR = (D+1)//2
_network = DDFtopo(V, D)
_network.pre_calculate_ECMP_ASP()
_network.pre_calculate_APST_n(4)
M_EPs_s = []
ECMP_ASP_phis = []
M_EPs_names=[]
# # # Define multiple traffic demand matrices:
# # shifts
for _shift in range(1, V*EPR):
# for _shift in range(1, 5):
    M_EPs = gl.generate_shift_traffic_pattern(V, EPR, _shift)
    # try to scale the traffic scaling factor to 10x saturation under ECMP_ASP
    core_link_flows, access_link_flows = _network.distribute_M_EPs_on_weighted_paths(_network.ECMP_ASP, EPR, M_EPs)
    max_core_link_load = np.max(core_link_flows)/Cap_core
    max_access_link_load = np.max(access_link_flows)/Cap_access
    traffic_scaling = 10.0/max(max_access_link_load, max_core_link_load)
    M_EPs = traffic_scaling * M_EPs
    # calculate phi for ECMP_ASP routing
    core_link_flows, access_link_flows = _network.distribute_M_EPs_on_weighted_paths(_network.ECMP_ASP, EPR, M_EPs)
    max_core_link_load = np.max(core_link_flows)/Cap_core
    max_access_link_load = np.max(access_link_flows)/Cap_access
    ECMP_ASP_phi=gl.network_total_throughput(M_EPs, max_core_link_load, max_access_link_load)/(V*EPR)
    ECMP_ASP_phis.append(ECMP_ASP_phi)
    # ==============

    # manage data
    M_EPs_s.append(M_EPs)
    # M_Rs.append(M_R)
    # max_access_link_loads.append(max_access_link_load)
    M_EPs_names.append(f"shift_{_shift}")

M_EPs_weights = [1/len(M_EPs_s) for _ in range(len(M_EPs_s))]




In [3]:
MD_container = nexullance_exp_container("DDF", V, D, EPR)

/users/ziyzhang/miniconda3/envs/gt/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/users/ziyzhang/miniconda3/envs/gt/lib/python3.12/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/users/ziyzhang/miniconda3/envs/gt/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/users/ziyzhang/miniconda3/envs/gt/lib/python3.12/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


In [6]:
MD_IT_result = MD_container.run_and_profile_MD_nexullance_IT(M_EPs_s, M_EPs_weights, repetitions=3)
    

In [7]:
MD_IT_result

{'ave_obj': 0.2895907759666443,
 'std_obj': 0.0,
 'ave_time[s]': 1.0472464443333334,
 'std_time[s]': 0.053579597790947904,
 'ave_PeakRAM[B]': 6626194.666666667,
 'std_PeakRAM[B]': 11476571.537702726}

In [8]:
MD_container.run_MD_nexullance_MP(M_EPs_s, M_EPs_weights, 4)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2411299
Academic license 2411299 - for non-commercial use only - registered to zi___@ugent.be
Set parameter Threads to value 1
Optimal solution found


(0.243959014418473,
 [10.0,
  8.815688871718146,
  5.877125914478764,
  5.873057171784216,
  5.868994058785019,
  5.686648253866497,
  5.368102804601929,
  4.94887127614622,
  4.5903775811512,
  4.271309761943254,
  3.9937147629632994,
  3.75,
  3.652979651162789,
  3.5608529328421636,
  3.394105135217268,
  3.5907067157005192,
  3.716375710034898,
  3.8511601584606834,
  3.9397642427212483,
  4.03254138571786,
  4.129793510324485,
  3.994927076727963,
  3.8685907276634874,
  3.749999999999991,
  3.775372124492552,
  3.801089918256128,
  3.8271604938271646,
  3.8334707337180296,
  3.8398018166803776,
  3.8461538461538245,
  3.860294117647043,
  3.8745387453874436,
  3.8888888888888857,
  3.8414634146341817,
  3.7951807228915384,
  3.750000000000015,
  3.80281690140847,
  3.857142857142704,
  3.913043478260899,
  3.956896551724176,
  4.001743679163002,
  4.047619047619049,
  4.020125786163562,
  3.9930034982508875,
  3.966244725738383,
  3.891444342226321,
  3.819413092550771,
  3.74999

In [4]:
MD_MP_result = MD_container.run_and_profile_MD_nexullance_MP(M_EPs_s, M_EPs_weights, 4, repetitions=1)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2411299
Academic license 2411299 - for non-commercial use only - registered to zi___@ugent.be
Set parameter Threads to value 1
Optimal solution found
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2411299
Academic license 2411299 - for non-commercial use only - registered to zi___@ugent.be
Set parameter Threads to value 1


In [ ]:
MD_MP_result